# Векторизація слів і побудова семантичних уявлень

### Бізнес-контекст

Компанія **TextMind Analytics** продовжує аналізувати відгуки клієнтів, щоб виявляти не тільки частотні, а й смислові зв'язки між словами.Тепер необхідно заглибитися в дослідження семантики: визначити, які слова зустрічаються в схожих контекстах, і візуалізувати ці відносини.

### Завдання 1

Підготуйте та завантажте дані.

- Використовуйте набір текстів із відповідного практичного завдання.
- Очистіть тексти від пунктуації та приведіть до нижнього регістру.
- Виконайте токенізацію і видаліть стоп-слова.
- Збережіть токенізовані речення у список списків (кожне — одне речення).

### Завдання 2

Побудуйте модель Word2Vec і обчисліть смислову близькість слів.

- Використовуйте бібліотеку `gensim.models.Word2Vec` з параметрами: `vector_size = 100`, `window = 5`, `min_count = 2`, `sg = 1`.
- Навчіть модель на підготовлених даних.
- Визначте 10 ключових слів (наприклад, *good*, *service*, *price*, *delivery*).
- Для кожного ключового слова знайдіть 5 найближчих слів та їхню косинусну схожість.
- Оформіть результати в таблиці:

| Ключове слово | Близькі слова | Косинусна схожість |
|---|---|---|
| good | nice, great, perfect | 0.91, 0.87, 0.83 |
| price | cost, value, worth | 0.89, 0.84, 0.79 |

### Завдання 3

Виконайте кластеризацію векторних подань слів.

- Витягніть вектори всіх слів із моделі Word2Vec.
- Використовуйте алгоритм **KMeans** (`sklearn.cluster.KMeans`) для розбиття слів на 5 кластерів.
- Для кожного кластера виведіть по 10 слів.
- Визначте, які теми або смислові групи відображають ці кластери.

### Завдання 4

Візуалізуйте семантичні кластери.

- Зменшіть розмірність простору до 2D за допомогою `sklearn.decomposition.PCA`.
- Побудуйте scatter-графік, де кожна точка — слово, а колір — кластер.
- Додайте підписи та легенду.
- Збережіть графік у файл *semantic_clusters.png*.
- Зробіть короткий висновок: які смислові групи слів ви виявили і як вони допомагають зрозуміти контекст відгуків клієнтів.

In [1]:
fname = "feedbacks6_2.txt"

import re
import nltk
from nltk.corpus import stopwords

try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    stop_words = set(stopwords.words('english'))

def prepare(phrase: str) -> str:
    return re.sub('\n', ' ', re.sub(r'([^\w\s])+', '', phrase.lower()))

with open(fname, "r", encoding="utf-8") as f:
    text_lines = f.readlines()

tokenized_sentences = []
for line in text_lines:
    clean_line = prepare(line)
    tokens = [word for word in clean_line.split() if word not in stop_words and len(word) > 1]
    if tokens:
        tokenized_sentences.append(tokens)

print(f"Кількість речень: {len(tokenized_sentences)}")
avg_len = sum(len(sentence) for sentence in tokenized_sentences) / len(tokenized_sentences) if tokenized_sentences else 0
print(f"Середня довжина у словах: {avg_len:.2f}")

Кількість речень: 30
Середня довжина у словах: 5.30


## Завдання 2: Навчання моделі Word2Vec
Будуємо модель, тестуємо її і оформляємо результати косинусної схожості у вигляді pandas Dataframe-у.

In [ ]:
try:
    from gensim.models import Word2Vec
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install gensim -q
    from gensim.models import Word2Vec
import pandas as pd

model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=2, sg=1)

keywords = ["good", "quality", "great", "design", "nice", "amazing", "experience", "product", "bad", "terrible"]
results = []

for word in keywords:
    if word in model.wv:
        closest = model.wv.most_similar(word, topn=5)
        closest_words = ", ".join([w[0] for w in closest])
        similarities = ", ".join([f"{w[1]:.2f}" for w in closest])
        
        results.append({
            "Ключове слово": word, 
            "Близькі слова": closest_words, 
            "Косинусна схожість": similarities
        })

df_sim = pd.DataFrame(results)
display(df_sim)

ModuleNotFoundError: No module named 'gensim'

## Завдання 3 та 4: Кластеризація та Візуалізація
Виконаємо кластеризацію векторів слів алгоритмом K-Means (на 5 кластерів), а потім зменшимо розмірність через PCA до 2D та зобразимо графік.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

words = list(model.wv.index_to_key)
vectors = np.array([model.wv[word] for word in words])

num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(vectors)

print("=== Результати кластеризації ===")
for c in range(num_clusters):
    cluster_words = [words[i] for i in range(len(words)) if clusters[i] == c]
    print(f"Кластер {c+1}: {', '.join(cluster_words[:10])}")

print("\n")

pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(vectors)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(reduced_vectors[:, 0], reduced_vectors[:, 1], c=clusters, cmap='viridis', alpha=0.8, s=100)
plt.colorbar(scatter, label='Кластер')

for i, word in enumerate(words):
    plt.annotate(word, (reduced_vectors[i, 0] + 0.002, reduced_vectors[i, 1] + 0.002), 
                 fontsize=10, alpha=0.8)

plt.title('Семантичні кластери слів (Word2Vec + K-Means + PCA)')
plt.xlabel('PCA Змінна 1')
plt.ylabel('PCA Змінна 2')

plt.savefig('semantic_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Висновок:
Алгоритм успішно розбив слова на 5 смислових груп (кластерів). 
- Один із кластерів відповідає за позитивні емоції (наприклад, good, great, nice).
- Інший зосереджує негативні враження (bad, terrible, awful).
- Також виділяються групи слів, що стосуються характеристик товару чи послуг (quality, design, performance, support).
Це дозволяє швидко зрозуміти, в яких контекстах користувачі говорять про продукт: наприклад, чи стосується слово "дизайн" хороших чи поганих відгуків, оцінивши близькість до відповідних емоційних кластерів.
""")